In the Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv we have infiltration attacks which are known to be stealthy. So instead of flooding the network with millions of packets (like the port scan was), an infiltration attack often starts with something more simpl: a single compromised machine inside the network, communicating quietly.

this means that the dataset will be mostly normal traffic (~99.9% as a reference). Because of this i will be using SMOTE (Synthetic Minority Over-sampling Technique). this one is used exactly for imbalanced datasets. It improves the performance by generating new examples. It selects a minority class sample and its nearest neighbors, then creates new synthetic points between them in the feature space, using linear interpolation to create new observations.

If we don't use smote, the model could just tell us that the accuracy is at 99.9 and it actually misses every single attack, which isn t something we want

A golden rule of this smote is that we need to be splitting our data into train and test BEFORE we use smote on it. If i apply SMOTE to the entire dataset first, synthetic "fake" attack data will leak into the testing set. We only oversample the Training data, as the Testing data must remain purely authentic to prove the model works in the real world

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [2]:
print(" Loading the Infiltration Dataset")
file_path = "MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv"
df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

print("\n The Imbalance Problem (Before Cleaning)")
print(df['Label'].value_counts())

#we can already notice a huge difference in the number of attacks with only 36 inflictions

 Loading the Infiltration Dataset

 The Imbalance Problem (Before Cleaning)
Label
BENIGN          288566
Infiltration        36
Name: count, dtype: int64


In [3]:
print("\n Cleaning and Encoding Data")
df_clean = df.dropna()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

df_clean['Label'] = df_clean['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print("\n The Imbalance Problem (After Cleaning)")
print(df_clean['Label'].value_counts())



 Cleaning and Encoding Data

 The Imbalance Problem (After Cleaning)
Label
0    288359
1        36
Name: count, dtype: int64


In [4]:
print("\n Splitting Data (The Golden Rule: Splitting BEFORE SMOTE)")
X = df_clean.drop('Label', axis=1)
y = df_clean['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Set Label Count BEFORE SMOTE:\n{y_train.value_counts()}")


 Splitting Data (The Golden Rule: Splitting BEFORE SMOTE)
Training Set Label Count BEFORE SMOTE:
Label
0    230686
1        30
Name: count, dtype: int64


In [5]:
print("\n Applying SMOTE to balance the Training Data")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Training Set Label Count AFTER SMOTE:\n{y_train_smote.value_counts()}")


 Applying SMOTE to balance the Training Data
Training Set Label Count AFTER SMOTE:
Label
0    230686
1    230686
Name: count, dtype: int64


In [6]:
print("\n Scaling the Data")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote) #fit on the SMOTE data
X_test_scaled = scaler.transform(X_test)             #transforming the authentic test data




 Scaling the Data


In [7]:
print("\n Multi-Model Cycling")
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=100, random_state=42)
}

results = []



 Multi-Model Cycling


In [8]:
for name, model in models.items():
    print(f"  -> Training {name}...")
    start_time = time.time()
    
    #trining on balanced and scaled smote
    model.fit(X_train_scaled, y_train_smote)
    
    #predicting on the untouched, unbalanced test data
    y_pred = model.predict(X_test_scaled)
    
    train_time = time.time() - start_time
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    #adding Recall here because it's crucial for rare attacks to know how many attacks did it actually catch
    recall = recall_score(y_test, y_pred) 
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Recall (Attack Catch Rate)": round(recall * 100, 4),
        "F1-Score": round(f1, 4),
        "Training Time (s)": round(train_time, 2)
    })

  -> Training Decision Tree...
  -> Training Random Forest...
  -> Training XGBoost...
  -> Training LightGBM...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Training Logistic Regression...
  -> Training MLP (Neural Net)...


In [9]:
print("\n Final Infiltration model leaderboard")

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
display(results_df)


 Final Infiltration model leaderboard


,Model,Accuracy (%),Recall (Attack Catch Rate),F1-Score,Training Time (s)
3,LightGBM,100.0000,100.0000,1.0000,9.65
2,XGBoost,99.9983,83.3333,0.9091,11.11
1,Random Forest,99.9965,66.6667,0.8000,33.16
0,Decision Tree,99.9948,66.6667,0.7273,31.43
5,MLP (Neural Net),99.9064,83.3333,0.1562,812.43
4,Logistic Regression,98.4951,100.0000,0.0136,21.68


This might be one of the more important little tables for my paper, as it shows quite a few differences.
We can again observe that MLP took a long time, almost boringly so, and it wasn t even good. This is one more proof on why for tabular data decision trees are better.

Now for the rest of the table:
Logistic Regression showed a recall catch of 100%, BUT if we look at the f1 score we can see how low it actually is. Because i used SMOTE, i had the training data be inflated with synthetic attacks. LR being mathematical got overly sensitive and most likely started flagging thousands of normal traffic as attacks. So even though it caught the attacks too, it was by pure luck. Irl a security analyst would turn this off immediately because of so many False-Positives. F1 balances recall and precision so it exposed our problem here very nicely.

And for a beautiful win: LightGBM:
It has 100% accuracy with no false-positives, which is exactly what we love seeing (even as it seems too good to be true). It learned the exact complex thresholds needed to spot the stealthy infiltration without triggering false alarms.

And now another look at the classics that worked well earlier for ddos and port scan, but now their accuracy dropped: Random Forest and Decision Tree
This does prove that the stealthy attacks are doing their job in staying hidden, as the basic models missed 1/3 of the attacks 